## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import multiprocessing
import multiprocessing.pool
import time
from pathlib import Path

import cv2
import imutils
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Checkbox, IntSlider, interact
from natsort import natsorted
from scipy import ndimage
from skimage import color, feature, filters, measure, segmentation

from pkg.circles import fit_circle_least_squares, fit_circle_RANSAC
from pkg.helpers import (
    crop_with_roi,
    get_output_vid_frame_size,
    get_roi_from_video,
    show_img,
)
from pkg.selection_window import SelectionWindow
from pkg.template_matching import get_template_matches, get_templates_from_video
from pkg.tracker import ReIDTracker

## RGB Thresholding

In [ ]:
img_ball = cv2.imread("data/ball/ball.jpg")
cv2.imshow("Original Image", img_ball)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
def show_selection(xPos, yPos, sqLen):
    frameSq = img_ball.copy()
    frameSq = cv2.rectangle(
        frameSq, (xPos, yPos), (xPos + sqLen, yPos + sqLen), (0, 255, 0), 1
    )

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frameSq, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

    selectedPixels = img_ball[yPos : yPos + sqLen, xPos : xPos + sqLen]
    pixelsMean = np.round(selectedPixels.mean(axis=(0, 1))).astype("int")
    print(f"Mean pixel values (BGR): {[int(c) for c in pixelsMean]}")


interact(
    show_selection,
    xPos=IntSlider(value=218, min=0, max=500, step=1, description="X Position:"),
    yPos=IntSlider(value=152, min=0, max=500, step=1, description="Y Position:"),
    sqLen=IntSlider(value=173, min=20, max=200, step=1, description="Square Size:"),
)

In [ ]:
def threshold_ball(delta=39, base_b=0, base_g=66, base_r=160, show_contours=True):
    base = np.array([base_b, base_g, base_r])
    boundary = [(base - delta).clip(min=0), (base + delta).clip(max=255)]

    lower, upper = np.array(boundary, dtype="uint8")
    mask = cv2.inRange(img_ball, lower, upper)

    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
    frame_cnts = img_ball.copy()

    if show_contours:
        cnts, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(frame_cnts, cnts, -1, (0, 255, 0), 1)

    comb = np.hstack([mask_rgb, frame_cnts])
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(comb, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


interact(
    threshold_ball,
    delta=IntSlider(value=39, min=0, max=120, step=1, description="Delta"),
    base_b=IntSlider(value=0, min=0, max=255, step=1, description="Base B"),
    base_g=IntSlider(value=66, min=0, max=255, step=1, description="Base G"),
    base_r=IntSlider(value=160, min=0, max=255, step=1, description="Base R"),
    show_contours=Checkbox(value=True, description="Draw contours"),
)

In [ ]:
delta = 101
baseH = 11
h_lo = np.clip(baseH - delta, a_min=0, a_max=180)
h_hi = np.clip(baseH + delta, a_min=0, a_max=180)

boundary = [[h_lo, 70, 0], [h_hi, 255, 255]]

lower, upper = np.array(boundary, dtype="uint8")

# For ball.JPG
frameHSV = cv2.cvtColor(img_ball, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(frameHSV, lower, upper)
output = cv2.bitwise_and(frameHSV, frameHSV, mask=mask)
maskRGB = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
frameCnts1 = img_ball.copy()
cnts, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(frameCnts1, cnts, -1, (255, 0, 0), 1)

# For ballLight.JPG
img_ball_light = cv2.imread("data/ball/ball_light.jpg")
frameHSV = cv2.cvtColor(img_ball_light, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(frameHSV, lower, upper)
output = cv2.bitwise_and(frameHSV, frameHSV, mask=mask)
maskRGB = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
frameCnts2 = img_ball_light.copy()
cnts, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(frameCnts2, cnts, -1, (255, 0, 0), 1)

show_img(frameCnts1)
plt.show()

show_img(frameCnts2)
plt.show()

## Gel Electrophoresis

In [ ]:
# Load image
imgPath = Path("data/gel_electrophoresis/gel_electrophoresis.tif")
img = cv2.imread(str(imgPath))

# Get region of interest
r = cv2.selectROI(img, fromCenter=False)
cv2.destroyAllWindows()
cropped = crop_with_roi(img, r)

plt.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
plt.axis("off")
# cv2.imwrite("data/cropped.jpg", cropped) # Uncomment to save cropped image

In [ ]:
cropped = cv2.imread("data/gel_electrophoresis/cropped.jpg")

# Convert to grayscale
gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

# Iteratively increase contrast
iterations = 8
clahe = cv2.createCLAHE(clipLimit=1.1, tileGridSize=(8, 8))
tophat1 = gray.copy()
for i in range(iterations):
    cl1 = clahe.apply(tophat1)
    tophat1 = cv2.morphologyEx(cl1, cv2.MORPH_TOPHAT, cl1)

plt.imshow(tophat1, cmap="gray")
plt.axis("off")
cv2.imwrite("data/gel_electrophoresis/enhanced.png", tophat1)

In [ ]:
# Morph to strips
# tophat1 = cv2.imread(
#     "data/gel_electrophoresis/gel_electrophoresis_enhanced.jpg", cv2.IMREAD_GRAYSCALE
# )

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (24, 1))
tophat2 = cv2.morphologyEx(tophat1, cv2.MORPH_OPEN, kernel)

plt.imshow(tophat2, cmap="gray")
plt.axis("off")
cv2.imwrite("data/gel_electrophoresis/enhanced2.png", tophat2)

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 2))
tophat3 = cv2.morphologyEx(tophat2, cv2.MORPH_OPEN, kernel)

plt.imshow(tophat3, cmap="gray")
plt.axis("off")
cv2.imwrite("data/gel_electrophoresis/enhanced3.png", tophat3)

In [ ]:
ret, th = cv2.threshold(tophat3, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.imshow(th, cmap="gray")
plt.axis("off")
cv2.imwrite("data/gel_electrophoresis/thresholded.png", th)

In [ ]:
cnts, _ = cv2.findContours(th, cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)
cropped_with_cnts = cv2.drawContours(cropped, cnts, -1, (0, 0, 255), 1)

plt.imshow(cv2.cvtColor(cropped_with_cnts, cv2.COLOR_BGR2RGB))
plt.axis("off")
cv2.imwrite("data/gel_electrophoresis/contours.png", cropped_with_cnts)

## Colony Counting

In [ ]:
img = cv2.imread(f"data/colony_counting/C_1.jpg")
img = imutils.resize(img, 600)
r = cv2.selectROI(img, fromCenter=False)
cv2.destroyAllWindows()
cropped = crop_with_roi(img, r)

plt.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
plt.axis("off")
# cv2.imwrite(f"data/colony_counting/C_1_cropped.jpg", cropped)

In [ ]:
cropped = cv2.imread("data/colony_counting/C_1_cropped.jpg")
gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(12, 12))
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
tophat1 = gray.copy()
cl1 = clahe.apply(tophat1)
tophat1 = cv2.morphologyEx(cl1, cv2.MORPH_TOPHAT, cl1)

plt.imshow(tophat1, cmap="gray")
plt.axis("off")
cv2.imwrite(f"data/colony_counting/C_1_CLAHE.png", tophat1)

In [ ]:
height, width = tophat1.shape
mask1 = np.zeros_like(tophat1)
cv2.circle(mask1, (width // 2, height // 2), round(0.35 * width), 255, -1)
tophat1_cropped = cv2.bitwise_and(tophat1, mask1)

plt.imshow(tophat1, cmap="gray")
plt.axis("off")
cv2.imwrite(f"data/colony_counting/C_1_CLAHE_cropped.png", tophat1)

In [ ]:
thresholds = filters.threshold_multiotsu(tophat1_cropped, classes=3)
regions = np.digitize(tophat1_cropped, bins=thresholds)

plt.imshow(regions)
plt.axis("off")
plt.savefig(f"data/colony_counting/C_1_regions.png", bbox_inches="tight", pad_inches=0)
plt.show()

cells = tophat1_cropped > thresholds[1]

plt.imshow(cells, cmap="gray")
plt.axis("off")
plt.savefig(f"data/colony_counting/C_1_cells.png", bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
# Label connected components to get distinct colors
cells_labeled = measure.label(cells)
plt.imshow(color.label2rgb(cells_labeled, bg_label=0))
plt.axis("off")
plt.savefig(
    f"data/colony_counting/C_1_cells_undercount.png", bbox_inches="tight", pad_inches=0
)
plt.show()

In [ ]:
distance = ndimage.distance_transform_edt(cells)

local_max_coords = feature.peak_local_max(distance, min_distance=1)
local_max_mask = np.zeros(distance.shape, dtype=bool)
local_max_mask[tuple(local_max_coords.T)] = True
markers = measure.label(local_max_mask)

segmented_cells = segmentation.watershed(-distance, markers, mask=cells)

plt.imshow(color.label2rgb(segmented_cells, bg_label=0))
plt.axis("off")
plt.savefig(
    f"data/colony_counting/C_1_cells_labelled.png", bbox_inches="tight", pad_inches=0
)
plt.show()

In [ ]:
color_labels = color.label2rgb(segmented_cells, cropped, alpha=0.4, bg_label=0)

numCells = segmented_cells.max()
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(color_labels)
circle1 = plt.Circle(
    (width // 2, height // 2), round(0.35 * width), color="black", fill=False
)
ax.add_artist(circle1)
ax.set_axis_off()
plt.savefig(
    f"data/colony_counting/C_1_cells_overlaid.png", bbox_inches="tight", pad_inches=0
)
plt.show()

## RANSAC

In [ ]:
img = cv2.imread("data/funnel/DSC08105.jpg")
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
low = np.array([5, 130, 140])
high = np.array([20, 255, 255])
img_th = cv2.inRange(img_hsv, low, high)
cnts, _ = cv2.findContours(img_th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
largest_cnt = max(cnts, key=cv2.contourArea)
img_with_cnt = cv2.drawContours(img, [largest_cnt], -1, (0, 255, 0), 3)
cnt_pts = largest_cnt.reshape(-1, 2)

show_img(img_with_cnt)
plt.show()

In [ ]:
(cx, cy, r) = fit_circle_least_squares(cnt_pts)
img_with_circle = cv2.circle(img.copy(), (int(cx), int(cy)), int(r), (255, 0, 0), 3)
show_img(img_with_circle)
plt.show()

In [ ]:
(cx, cy, r), inlier_pts = fit_circle_RANSAC(cnt_pts, 1000, 3)
img_with_circle = cv2.circle(img, (int(cx), int(cy)), int(r), (255, 0, 0), 3)
show_img(img_with_circle)

plt.scatter(cnt_pts[:, 0], cnt_pts[:, 1], color="red", s=1)
plt.scatter(inlier_pts[:, 0], inlier_pts[:, 1], color="green", s=1)

## Template Matching

### Get Crop

In [ ]:
vidPath = "data/pancake/video.mov"
roi = get_roi_from_video(vidPath)

In [ ]:
vid_path = "data/pancake/video.mov"
output_vid_path = "data/pancake/video_cropped.mp4"
output_height = 800

cap = cv2.VideoCapture(vid_path)

pipeline = lambda frame: crop_with_roi(frame, roi)

frameWidth, frameHeight = get_output_vid_frame_size(vid_path, pipeline, output_height)
out = cv2.VideoWriter(
    output_vid_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    cap.get(cv2.CAP_PROP_FPS),
    (frameWidth, frameHeight),
)

ret, frame = cap.read()
while ret:
    cropped_frame = crop_with_roi(frame, roi)
    cropped_frame = imutils.resize(cropped_frame, height=output_height)

    out.write(cropped_frame)
    ret, frame = cap.read()

cap.release()
out.release()

### Get Template(s)

In [ ]:
pipeline = lambda frame: crop_with_roi(frame, roi)
templates = get_templates_from_video(
    vidPath, pipeline, templateWidth=100, templateHeight=100
)

### Save Template(s)

In [ ]:
templatesDir = Path("data/pancake/templates/grey")
imgPaths = natsorted([str(path) for path in templatesDir.glob("*.jpg")])

if imgPaths:
    latestImgN = int(Path(imgPaths[-1]).stem) + 1
else:
    latestImgN = 0

for i in range(len(templates)):
    cv2.imwrite(str(templatesDir / f"{latestImgN+i}.jpg"), templates[i])

templates.clear()

### Load Template(s)

In [ ]:
templatesDir = Path("data/pancake/templates/grey/")
imgPaths = templatesDir.glob("*.jpg")

templates = []
for imgPath in imgPaths:
    template = cv2.imread(str(imgPath))
    templates.append(template)
    show_img(template)
    plt.show()

### Match Template(s)

In [ ]:
VID_PATH = "data/pancake/video.mov"
OUTPUT_VID_PATH = "data/pancake/video_output.mp4"

OUTPUT_HEIGHT = 800

NMS_THRESHOLD = 0.7
CONFIDENCE_THRESHOLD = 0.775

num_cpu = multiprocessing.cpu_count() - 1
pool = multiprocessing.pool.ThreadPool(processes=num_cpu)

cap = cv2.VideoCapture(VID_PATH)

totalFrames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
frameCount = 0
startTime = time.time()
pipeline = lambda frame: crop_with_roi(frame, roi)

frameWidth, frameHeight = get_output_vid_frame_size(VID_PATH, pipeline, OUTPUT_HEIGHT)
out = cv2.VideoWriter(
    OUTPUT_VID_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    cap.get(cv2.CAP_PROP_FPS),
    (frameWidth, frameHeight),
)
print(f"Output frame width: {frameWidth}, frame height: {frameHeight}")

saveFrames = []
saveBoxes = []

ret, frame = cap.read()
while ret:
    if frameCount % 100 == 0 and frameCount != 0:
        elapsedTime = time.time() - startTime
        estTimeLeft = (totalFrames - frameCount) / frameCount * elapsedTime
        print(f"Frame {frameCount} out of {round(totalFrames)}.")
        print(
            f"\tTime taken: {round(elapsedTime)}s. Est. time left: {round(estTimeLeft)}s"
        )

    img_rgb = pipeline(frame)

    # Multithreading
    mapIterable = []
    for i in range(len(templates)):
        template = templates[i]
        mapIterable.append((img_rgb, template, CONFIDENCE_THRESHOLD))

    results = pool.starmap(func=get_template_matches, iterable=mapIterable)

    boxes, confidences = [], []
    for result in results:
        boxes.extend(result[0])
        confidences.extend(result[1])

    indices = cv2.dnn.NMSBoxes(boxes, confidences, CONFIDENCE_THRESHOLD, NMS_THRESHOLD)
    boxes = [boxes[idx] for idx in indices]
    confidences = [confidences[idx] for idx in indices]

    if boxes:
        saveFrames.append(frameCount)
        saveBoxes.append(boxes[0].copy())

    for box in boxes:
        cv2.rectangle(img_rgb, box[:2], box[2:], (0, 0, 255), 2)

    img_rgb = imutils.resize(img_rgb, height=OUTPUT_HEIGHT)

    out.write(img_rgb)

    cv2.imshow("Detections", img_rgb)
    key = cv2.waitKey(1)
    if key == ord("q") or key == ord("Q"):
        break

    ret, frame = cap.read()
    frameCount += 1

cv2.destroyAllWindows()
out.release()
cap.release()
pool.close()
pool.join()

### Match Template(s) with ReID Tracking

In [ ]:
VID_PATH = "data/pancake/video.mov"
OUTPUT_VID_PATH = "data/pancake/video_output_reid.mp4"

OUTPUT_HEIGHT = 800

NMS_THRESHOLD = 0.7
CONFIDENCE_THRESHOLD = 0.775

tracker = ReIDTracker(max_distance=100, max_frames_to_skip=30)

num_cpu = multiprocessing.cpu_count() - 1
pool = multiprocessing.pool.ThreadPool(processes=num_cpu)

cap = cv2.VideoCapture(VID_PATH)

totalFrames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
frameCount = 0
startTime = time.time()
pipeline = lambda frame: crop_with_roi(frame, roi)

frameWidth, frameHeight = get_output_vid_frame_size(VID_PATH, pipeline, OUTPUT_HEIGHT)
out = cv2.VideoWriter(
    OUTPUT_VID_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    cap.get(cv2.CAP_PROP_FPS),
    (frameWidth, frameHeight),
)
print(f"Output frame width: {frameWidth}, frame height: {frameHeight}")

# Track history for visualization
track_history = {}

ret, frame = cap.read()
while ret:
    if frameCount % 100 == 0 and frameCount != 0:
        elapsedTime = time.time() - startTime
        estTimeLeft = (totalFrames - frameCount) / frameCount * elapsedTime
        print(f"Frame {frameCount} out of {round(totalFrames)}.")
        print(
            f"\tTime taken: {round(elapsedTime)}s. Est. time left: {round(estTimeLeft)}s"
        )

    img_rgb = pipeline(frame)

    mapIterable = []
    for i in range(len(templates)):
        template = templates[i]
        mapIterable.append((img_rgb, template, CONFIDENCE_THRESHOLD))

    results = pool.starmap(func=get_template_matches, iterable=mapIterable)

    boxes, confidences = [], []
    for result in results:
        boxes.extend(result[0])
        confidences.extend(result[1])

    if boxes:
        indices = cv2.dnn.NMSBoxes(
            boxes, confidences, CONFIDENCE_THRESHOLD, NMS_THRESHOLD
        )
        boxes = [boxes[idx] for idx in indices]
        confidences = [confidences[idx] for idx in indices]

    tracks = tracker.update(img_rgb, boxes)

    # Draw tracked objects with IDs
    for track_id, track in tracks.items():
        box = track["box"]
        x1, y1, x2, y2 = box

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        if track_id not in track_history:
            track_history[track_id] = []
        track_history[track_id].append((cx, cy))

        # Keep only last 30 points
        if len(track_history[track_id]) > 30:
            track_history[track_id] = track_history[track_id][-30:]

        color = ((track_id * 67) % 255, (track_id * 113) % 255, (track_id * 191) % 255)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)

        # Draw ID label
        label = f"ID: {track_id}"
        label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(
            img_rgb, (x1, y1 - label_size[1] - 10), (x1 + label_size[0], y1), color, -1
        )
        cv2.putText(
            img_rgb,
            label,
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
        )

        # Draw trajectory
        points = track_history[track_id]
        for i in range(1, len(points)):
            thickness = int(np.sqrt(30 / float(i + 1)) * 2)
            cv2.line(img_rgb, points[i - 1], points[i], color, thickness)

    img_rgb = imutils.resize(img_rgb, height=OUTPUT_HEIGHT)

    out.write(img_rgb)

    cv2.imshow("ReID Tracking", img_rgb)
    key = cv2.waitKey(1)
    if key == ord("q") or key == ord("Q"):
        break

    ret, frame = cap.read()
    frameCount += 1

cv2.destroyAllWindows()
out.release()
cap.release()
pool.close()
pool.join()

print(f"\nTotal unique tracks: {tracker.next_track_id - 1}")

## Perspective Transform

In [ ]:
img = cv2.imread("data/screw/frame1000.png")
show_img(img)
plt.show()

In [ ]:
perspectiveWindow = SelectionWindow("Perspective", img)
perspectiveWindow.displayWindow()

In [ ]:
objLength = 1.83
objWidth = 0.6
imgWidth = 200
imgHeight = round(objLength / objWidth * imgWidth)

srcPts = np.float32(perspectiveWindow.selectionPts)
dstPts = np.float32([(0, 0), (imgWidth, 0), (imgWidth, imgHeight), (0, imgHeight)])
M = cv2.getPerspectiveTransform(srcPts, dstPts)
dst = cv2.warpPerspective(img, M, (imgWidth, imgHeight))

plt.imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()